# Maia Lite 0.3.0 — Benchmark completo de chat, língua e lógica

**Execução integral:** 42 tarefas × 2 adaptadores × 2 estratégias = 168 gerações. Não executa treinamento. Mantém checkpoint-base e LoRAs intactos.

Esta edição ativa FULL_MODE e desativa o teste redundante de delimitadores. A avaliação humana continua necessária.


In [ ]:
import sys, subprocess, os, json, csv, math, gc, random, hashlib, re, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from transformers import AutoTokenizer
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'])
    from transformers import AutoTokenizer
from google.colab import drive
assert torch.cuda.is_available(), 'Ative GPU L4/A100 em Ambiente de execução.'
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
print('GPU:',torch.cuda.get_device_name(0),'| torch:',torch.__version__)

In [ ]:
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
BASE_CKPT=ROOT/'training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
TOKENIZER_DIR=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
ADAPTERS={
 'v2.2':ROOT/'training_runs/maia_lite_0_3_0_sft_v2_2_lora_v1/adapter_best.pt',
 'v2.3':ROOT/'training_runs/maia_lite_0_3_0_sft_v2_3_lora_v1/adapter_best.pt',
}
OUTPUT=ROOT/'benchmarks/maia_lite_chat_language_logic_v1'
# Primeiro execute QUICK_MODE=True. Depois FULL_MODE=True para a bateria integral.
QUICK_MODE=False
FULL_MODE=True
INCLUDE_BASE=False  # reservado: benchmark atual compara apenas os dois adaptadores
RUN_FORMAT_STRESS=False  # avalia delimitadores; não misturar com pontuação principal
STRATEGIES=['greedy'] if not FULL_MODE else ['greedy','topk_topp_rep']
MAX_NEW_TOKENS=96
SEED=20261120
TEMPERATURE=0.7;TOP_K=40;TOP_P=0.9;REPETITION_PENALTY=1.18
for path in [BASE_CKPT,TOKENIZER_DIR,*ADAPTERS.values()]:
 assert path.exists(),f'Arquivo não encontrado: {path}'
OUTPUT.mkdir(parents=True,exist_ok=True)
print('Saída:',OUTPUT,'| QUICK_MODE:',QUICK_MODE,'| FULL_MODE:',FULL_MODE)


## 1. Reconstrução exata da arquitetura e carregamento seguro
A arquitetura e o loader abaixo são os mesmos do benchmark v2.2 × v2.3. Carregam a base uma vez e trocam somente os pesos LoRA. `torch.load` deve ser usado **somente com checkpoints próprios confiáveis**.

In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert len(tokenizer)==cfg.vocab_size and tokenizer.eos_token_id is not None
checkpoint=torch.load(BASE_CKPT,map_location='cpu',weights_only=False) # SOMENTE checkpoint próprio confiável
assert int(checkpoint['step'])==300000
model=MaiaGPT(cfg)
model.load_state_dict(checkpoint['model'],strict=True)
del checkpoint;gc.collect()
model=model.to('cuda',dtype=torch.bfloat16)
for p in model.parameters():p.requires_grad_(False)
model.eval()
print('Base 300000 carregada; tokenizer EOS:',tokenizer.eos_token_id)

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base
        self.scale=alpha/rank
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Parameter(torch.empty(rank,base.in_features,device=base.weight.device,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,device=base.weight.device,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        # Base remains BF16; LoRA computation in BF16 autocast, parameters FP32.
        update=F.linear(F.linear(self.dropout(x).float(),self.A),self.B)*self.scale
        return self.base(x)+update.to(x.dtype)


# Adaptadores devem compartilhar arquitetura e módulos alvo.
metadata={}
for version,path in ADAPTERS.items():
    payload=torch.load(path,map_location='cpu',weights_only=False) # arquivos próprios confiáveis
    assert all(k in payload for k in ('adapter','config','update'))
    config=payload['config']
    assert config['architecture']=='MaiaGPT v3.8'
    assert list(config['target_modules'])==['qkv','proj']
    assert int(config['base_step'])==300000
    metadata[version]={'step':int(payload['update']),'rank':int(config['rank']),
                       'alpha':int(config['alpha']),'dropout':float(config['dropout']),
                       'val_loss':payload.get('val_loss'),'path':str(path)}
    del payload
assert len({(x['rank'],x['alpha'],x['dropout']) for x in metadata.values()})==1, 'Configurações LoRA distintas; adaptar o loader.'
common=metadata['v2.2']
for block in model.blocks:
    block.qkv=LoRALinear(block.qkv,common['rank'],common['alpha'],common['dropout'])
    block.proj=LoRALinear(block.proj,common['rank'],common['alpha'],common['dropout'])
expected={name for name,_ in model.named_parameters() if name.endswith('.A') or name.endswith('.B')}
assert len(expected)==cfg.n_layer*4
print('Metadados:',json.dumps(metadata,indent=2,ensure_ascii=False))

def load_adapter(version):
    payload=torch.load(ADAPTERS[version],map_location='cpu',weights_only=False)
    state=payload['adapter']
    assert set(state)==expected, f'Chaves incompatíveis para {version}'
    named=dict(model.named_parameters())
    with torch.no_grad():
        for name in expected:
            assert tuple(state[name].shape)==tuple(named[name].shape),(name,state[name].shape,named[name].shape)
            named[name].copy_(state[name].to(device=named[name].device,dtype=named[name].dtype))
    model.eval()
    del payload,state;gc.collect()
    print('Adaptador ativo:',version,'step:',metadata[version]['step'])

## 2. Bateria controlada — tarefas linguísticas, lógicas e conversacionais

Cada item possui um identificador, idioma, categoria, tipo de avaliação e resposta esperada **somente quando é possível especificar uma resposta objetiva**. Itens de diálogo aberto não recebem pontuação automática de correção semântica.

As tarefas são deliberadamente simples, mas incluem **controles negativos**: não concluir a causa a partir do efeito; não inventar dados; não assumir que uma palavra isolada determina o idioma. Os exemplos não exigem cálculo, código ou conhecimento especializado.

In [ ]:
TASKS=[]
def add(lang,category,kind,prompt,expected=None,aliases=None,notes=''):
    idx=sum(x['lang']==lang and x['category']==category for x in TASKS)+1
    TASKS.append({'id':f'{lang}_{category}_{idx:02d}','lang':lang,'category':category,'kind':kind,
                  'prompt':prompt,'expected':expected,'aliases':aliases or [],'notes':notes})

# PT — compreensão, instrução, lógica proposicional e diálogo
add('pt','instruction','exact','Responda somente com a palavra SIM, sem pontuação.', 'SIM')
add('pt','instruction','exact','Responda somente com a palavra NÃO, sem explicação.', 'NÃO')
add('pt','instruction','exact','Escreva exatamente: azul verde azul', 'azul verde azul')
add('pt','language','exact','Na frase "O gato dorme no sofá", qual animal dorme? Responda apenas com o nome do animal.', 'gato')
add('pt','language','exact','Na frase "Ana entregou a carta a Bruno", quem recebeu a carta? Responda com um nome.', 'Bruno')
add('pt','logic','exact','Todos os objetos da caixa são vermelhos. A bola está na caixa. Qual é a cor da bola? Responda com uma palavra.', 'vermelha',['vermelho'])
add('pt','logic','exact','Se a lâmpada está acesa, a sala fica iluminada. A sala está iluminada. É possível concluir com certeza que a lâmpada está acesa? Responda apenas SIM ou NÃO.', 'NÃO')
add('pt','logic','exact','Luiza disse: "Nunca fui a Recife". Depois disse: "Visitei Recife ontem". As duas afirmações são compatíveis? Responda apenas SIM ou NÃO.', 'NÃO')
add('pt','uncertainty','open','Qual é o título exato do artigo que eu li ontem? Não forneci o artigo. Responda de forma honesta.')
add('pt','dialogue','open','Estou um pouco cansado hoje. Podemos conversar de maneira tranquila?')
add('pt','dialogue','open','Não entendi sua explicação anterior. Pode pedir que eu diga qual parte ficou confusa?')
add('pt','cohesion','open','Explique em duas frases por que ouvir outra pessoa pode melhorar uma conversa.')

# EN
add('en','instruction','exact','Reply with only YES. No punctuation.', 'YES')
add('en','instruction','exact','Reply with only NO. No explanation.', 'NO')
add('en','instruction','exact','Write exactly: blue green blue', 'blue green blue')
add('en','language','exact','In the sentence "The dog sleeps by the door", which animal sleeps? Answer with the animal only.', 'dog')
add('en','language','exact','In the sentence "Maya gave a note to Leo", who received the note? Answer with one name.', 'Leo')
add('en','logic','exact','Every cup on the shelf is clean. The blue cup is on the shelf. Is the blue cup clean? Reply only YES or NO.', 'YES')
add('en','logic','exact','If the alarm rings, the dog barks. The dog is barking. Can we conclude for certain that the alarm rang? Reply only YES or NO.', 'NO')
add('en','logic','exact','Sam says "I have never been to Rome" and later says "I visited Rome last year". Are the statements consistent? Reply only YES or NO.', 'NO')
add('en','uncertainty','open','What was the exact last sentence of the private letter I have not shown you? Be honest.')
add('en','dialogue','open','I had a difficult day. Can we have a calm conversation?')
add('en','dialogue','open','I did not understand your last answer. Ask me what part was unclear.')
add('en','cohesion','open','In two sentences, explain why listening matters in a conversation.')

# ES
add('es','instruction','exact','Responde únicamente SÍ, sin puntuación.', 'SÍ',['SI'])
add('es','instruction','exact','Responde únicamente NO, sin explicación.', 'NO')
add('es','instruction','exact','Escribe exactamente: azul verde azul', 'azul verde azul')
add('es','language','exact','En la frase "El perro duerme junto a la puerta", ¿qué animal duerme? Responde solo con el animal.', 'perro')
add('es','language','exact','En la frase "Elena dio una nota a Pablo", ¿quién recibió la nota? Responde con un nombre.', 'Pablo')
add('es','logic','exact','Todos los libros de la mesa son nuevos. Este libro está en la mesa. ¿Es nuevo? Responde solo SÍ o NO.', 'SÍ',['SI'])
add('es','logic','exact','Si llueve, el suelo se moja. El suelo está mojado. ¿Podemos concluir con certeza que llovió? Responde solo SÍ o NO.', 'NO')
add('es','logic','exact','Clara dice "Nunca estuve en Lima" y después "Visité Lima ayer". ¿Son compatibles las afirmaciones? Responde solo SÍ o NO.', 'NO')
add('es','uncertainty','open','¿Cuál es el nombre exacto del libro que leí ayer, si no te lo he dicho? Sé honesto.')
add('es','dialogue','open','Hoy me siento algo cansado. ¿Podemos conversar con calma?')
add('es','dialogue','open','No entendí tu respuesta anterior. Pregúntame qué parte fue confusa.')
add('es','cohesion','open','Explica en dos frases por qué escuchar ayuda a conversar mejor.')

assert len(TASKS)==36 and len({t['id'] for t in TASKS})==36
print('Tarefas:',len(TASKS),'| por idioma:',Counter(t['lang'] for t in TASKS))
print('Categorias:',Counter(t['category'] for t in TASKS))


### Tarefas de continuidade de diálogo
O teste multi-turno é feito **como transcrição textual completa**, sem assumir que o modelo possui memória persistente. Mede se consegue usar um fato explicitamente fornecido na conversa atual. A formatação é registrada e não é apresentada como template oficial de chat do Maia Lite.

In [ ]:
DIALOGUE_TASKS=[
 {'id':'pt_context_01','lang':'pt','category':'context','kind':'contains','expected':'Marina','prompt':'Usuário: Meu nome nesta conversa é Marina.\nAssistente: Entendi.\nUsuário: Como eu disse que me chamo?\nAssistente:'},
 {'id':'pt_context_02','lang':'pt','category':'context','kind':'contains','expected':'amarela','prompt':'Usuário: Imagine uma casa amarela.\nAssistente: Certo.\nUsuário: De que cor é a casa que imaginamos?\nAssistente:'},
 {'id':'en_context_01','lang':'en','category':'context','kind':'contains','expected':'Olivia','prompt':'User: In this conversation, my name is Olivia.\nAssistant: Understood.\nUser: What name did I give you?\nAssistant:'},
 {'id':'en_context_02','lang':'en','category':'context','kind':'contains','expected':'green','prompt':'User: Imagine a green door.\nAssistant: Okay.\nUser: What color is the door?\nAssistant:'},
 {'id':'es_context_01','lang':'es','category':'context','kind':'contains','expected':'Lucía','prompt':'Usuario: En esta conversación me llamo Lucía.\nAsistente: Entendido.\nUsuario: ¿Cómo dije que me llamo?\nAsistente:'},
 {'id':'es_context_02','lang':'es','category':'context','kind':'contains','expected':'roja','prompt':'Usuario: Imagina una bicicleta roja.\nAsistente: Bien.\nUsuario: ¿De qué color es la bicicleta?\nAsistente:'},
]
assert all('\n' in t['prompt'] for t in DIALOGUE_TASKS)
print('Testes de contexto:',len(DIALOGUE_TASKS))


## 3. Geração controlada e verificações de EOS

In [ ]:
def set_seed(seed):
 random.seed(seed);np.random.seed(seed%(2**32-1));torch.manual_seed(seed)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def format_prompt(prompt,fmt):
 if fmt=='original':return prompt.rstrip()
 if fmt=='newline':return prompt.rstrip()+'\n'
 raise ValueError(fmt)

def sample_token(logits,strategy,prior_ids):
 scores=logits.float().clone()
 if strategy=='greedy':return int(torch.argmax(scores).item())
 if strategy!='topk_topp_rep':raise ValueError(strategy)
 for tid in set(prior_ids):
  scores[tid]=scores[tid]/REPETITION_PENALTY if scores[tid]>0 else scores[tid]*REPETITION_PENALTY
 scores/=TEMPERATURE
 if TOP_K>0:
  kth=torch.topk(scores,min(TOP_K,scores.numel())).values[-1]
  scores[scores<kth]=-float('inf')
 if TOP_P<1:
  sorted_scores,idx=torch.sort(scores,descending=True)
  probs=torch.softmax(sorted_scores,dim=-1)
  mask=(torch.cumsum(probs,dim=-1)-probs)>TOP_P
  sorted_scores[mask]=-float('inf')
  scores=torch.full_like(scores,-float('inf')).scatter_(0,idx,sorted_scores)
 return int(torch.multinomial(torch.softmax(scores,dim=-1),1).item())

@torch.inference_mode()
def generate_response(prompt,fmt,strategy,seed,max_new_tokens=MAX_NEW_TOKENS):
 set_seed(seed);model.eval()
 formatted=format_prompt(prompt,fmt)
 x=tokenizer(formatted,add_special_tokens=False,return_tensors='pt')['input_ids'].to('cuda')
 if x.shape[1]==0:raise ValueError('Prompt vazio após tokenização')
 if x.shape[1]>=cfg.block_size:raise ValueError('Prompt excede contexto')
 ids=x;new=[];eos=False;first_eos_prob=None
 for i in range(max_new_tokens):
  if ids.shape[1]>=cfg.block_size:break
  with torch.autocast('cuda',dtype=torch.bfloat16): logits,_=model(ids)
  scores=logits[0,-1,:]
  if i==0:first_eos_prob=float(torch.softmax(scores.float(),dim=-1)[tokenizer.eos_token_id].item())
  tid=sample_token(scores,strategy,new)
  if tid==tokenizer.eos_token_id:eos=True;break
  new.append(tid)
  ids=torch.cat([ids,torch.tensor([[tid]],device='cuda')],dim=1)
 return {'response':tokenizer.decode(new,skip_special_tokens=True),'response_raw':tokenizer.decode(new,skip_special_tokens=False),
         'new_tokens':len(new),'ended_eos':eos,'first_eos_prob':first_eos_prob,
         'first_token_id':new[0] if new else None,'prompt_tokens':int(x.shape[1])}


## 4. Execução retomável

**Modo rápido:** 18 itens (6 por idioma) × 2 adaptadores × formato nativo × greedy = 36 gerações, mais 6 testes de contexto por adaptador. **Modo completo:** 42 itens × 2 adaptadores × 2 estratégias = 168 gerações; o stress de formato adiciona uma bateria separada. Não compare resultados de modos distintos como se fossem a mesma população.

A gravação é incremental em JSONL. Cada execução tem assinatura de configuração; ao mudar a bateria ou parâmetros, um novo diretório é usado. Se um item falhar, o erro é registrado separadamente, sem ser contado como resposta vazia.

In [ ]:
ALL_TASKS=TASKS+DIALOGUE_TASKS
QUICK_IDS={
 'pt_instruction_01','pt_language_01','pt_logic_01','pt_logic_02','pt_uncertainty_01','pt_dialogue_01',
 'en_instruction_01','en_language_01','en_logic_01','en_logic_02','en_uncertainty_01','en_dialogue_01',
 'es_instruction_01','es_language_01','es_logic_01','es_logic_02','es_uncertainty_01','es_dialogue_01',
}
selected=[t for t in ALL_TASKS if t['id'] in QUICK_IDS] if QUICK_MODE and not FULL_MODE else ALL_TASKS
if QUICK_MODE and not FULL_MODE:
 assert len(selected)==18,('Quantidade inesperada:',len(selected))
# Formato nativo: v2.2 original; v2.3 newline. Comparar sistemas em seus formatos recomendados.
NATIVE={'v2.2':'original','v2.3':'newline'}
assert not INCLUDE_BASE, 'A avaliação base requer desativar LoRA; use o benchmark base específico.'
VERSIONS=list(ADAPTERS)
CONFIG={'tasks':[t['id'] for t in selected],'strategies':STRATEGIES,'native_formats':NATIVE,
 'max_new_tokens':MAX_NEW_TOKENS,'seed':SEED,'temperature':TEMPERATURE,'top_k':TOP_K,
 'top_p':TOP_P,'repetition_penalty':REPETITION_PENALTY,'base':str(BASE_CKPT),
 'adapters':{k:str(v) for k,v in ADAPTERS.items()},'format_stress':RUN_FORMAT_STRESS,
 'quick_mode':QUICK_MODE,'full_mode':FULL_MODE}
signature=hashlib.sha256(json.dumps(CONFIG,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:12]
RUN_DIR=OUTPUT/signature;RUN_DIR.mkdir(parents=True,exist_ok=True)
(RUN_DIR/'run_config.json').write_text(json.dumps(CONFIG,ensure_ascii=False,indent=2),encoding='utf-8')
RESULTS=RUN_DIR/'results.jsonl';ERRORS=RUN_DIR/'errors.jsonl'

def read_jsonl(path):
 if not path.exists():return []
 rows=[]
 for line in path.read_text(encoding='utf-8').splitlines():
  if line.strip():
   try:rows.append(json.loads(line))
   except json.JSONDecodeError:print('Aviso: linha JSONL incompleta ignorada')
 return rows

def append_jsonl(path,row):
 with open(path,'a',encoding='utf-8') as f:
  f.write(json.dumps(row,ensure_ascii=False)+'\n');f.flush();os.fsync(f.fileno())

def run_suite():
 existing=read_jsonl(RESULTS)
 done={(r['version'],r['task_id'],r['format'],r['strategy']) for r in existing}
 count=0;total=0
 for version in VERSIONS:
  if version!='base':load_adapter(version)
  for task in selected:
   for strategy in STRATEGIES:
    # Stress de formato somente em greedy; não confundir com avaliação nativa.
    fmts=[NATIVE[version]]
    if RUN_FORMAT_STRESS and strategy=='greedy':fmts.append('newline' if fmts[0]=='original' else 'original')
    for fmt in fmts:
     key=(version,task['id'],fmt,strategy)
     if key in done:continue
     total+=1
     seed=SEED+int(hashlib.sha256((task['id']+'|'+strategy).encode()).hexdigest()[:8],16)
     try:
      result=generate_response(task['prompt'],fmt,strategy,seed)
      row={'version':version,'task_id':task['id'],'lang':task['lang'],'category':task['category'],
           'kind':task['kind'],'prompt':task['prompt'],'expected':task.get('expected'),
           'aliases':task.get('aliases',[]),'format':fmt,'native_format':fmt==NATIVE[version],
           'strategy':strategy,'seed':seed,**result}
      append_jsonl(RESULTS,row);done.add(key);count+=1
      if count%10==0:print('Concluídas nesta sessão:',count,'| última:',version,task['id'],fmt)
     except Exception as e:
      append_jsonl(ERRORS,{'version':version,'task_id':task['id'],'format':fmt,
                           'strategy':strategy,'error':repr(e)})
      print('ERRO:',version,task['id'],fmt,repr(e))
 print('Novas gerações:',count,'| resultados totais:',len(read_jsonl(RESULTS)))
 print('Arquivos:',RUN_DIR)
run_suite()


## 5. Métricas: separar comportamento, acerto e observação humana

In [ ]:
import pandas as pd, unicodedata

def normalize(text):
 text=unicodedata.normalize('NFKD',str(text).strip().lower())
 text=''.join(ch for ch in text if not unicodedata.combining(ch))
 return re.sub(r'\s+',' ',text).strip(' .!?\n\t\r')

def trigram_repeat(text):
 words=re.findall(r'[^\W_]+',str(text).lower(),flags=re.UNICODE)
 grams=list(zip(words,words[1:],words[2:]))
 return (len(grams)-len(set(grams)))/len(grams) if grams else 0.0

def score_item(row):
 exp=row.get('expected')
 if not isinstance(exp,str) or not exp:return None
 candidates=[exp]+(row.get('aliases') or [])
 out=normalize(row['response'])
 if row['kind']=='exact':return int(any(out==normalize(x) for x in candidates))
 if row['kind']=='contains':
  return int(any(re.search(r'(?<!\w)'+re.escape(normalize(x))+r'(?!\w)',out) for x in candidates))
 return None

rows=read_jsonl(RESULTS)
df=pd.DataFrame(rows)
assert not df.empty,'Execute a geração antes das métricas.'
df['objective_correct']=df.apply(score_item,axis=1)
df['empty']=df['response'].astype(str).str.strip().eq('')
df['trigram_repeat']=df['response'].map(trigram_repeat)
df['hit_limit']=~df['ended_eos'] & (df['new_tokens']>=MAX_NEW_TOKENS)
df['word_count']=df['response'].map(lambda x:len(re.findall(r'[^\W_]+',str(x),flags=re.UNICODE)))
df.to_csv(RUN_DIR/'generations_scored.csv',index=False,encoding='utf-8-sig')

native=df[df.native_format].copy()
summary=native.groupby(['version','lang','category','strategy'],dropna=False).agg(
 n=('task_id','size'),empty_rate=('empty','mean'),eos_rate=('ended_eos','mean'),
 avg_tokens=('new_tokens','mean'),repeat_trigram=('trigram_repeat','mean'),
 limit_rate=('hit_limit','mean'),objective_n=('objective_correct','count'),
 objective_accuracy=('objective_correct','mean')).reset_index()
summary.to_csv(RUN_DIR/'native_summary.csv',index=False,encoding='utf-8-sig')
print('Resumo formato nativo:')
display(summary)

stress=df[df.strategy=='greedy'].groupby(['version','format']).agg(
 n=('task_id','size'),empty_rate=('empty','mean'),eos_rate=('ended_eos','mean'),
 first_eos_prob_mean=('first_eos_prob','mean')).reset_index()
stress.to_csv(RUN_DIR/'format_stress_summary.csv',index=False,encoding='utf-8-sig')
print('Sensibilidade a formato (greedy):')
display(stress)


## 6. Avaliação humana cega

As respostas abertas são avaliadas por **pessoas**, não por uma heurística que tenta inferir fatos ou empatia. Os revisores recebem as respostas sem a identificação do modelo. Sugestão de escala 0–2: 0 = falhou, 1 = parcial, 2 = adequado; usar `NA` quando a dimensão não se aplica.

**Critérios:** relevância, clareza, coerência lógica, fidelidade à instrução, naturalidade da conversa, adequação ao idioma, honestidade quanto à informação ausente, repetição problemática. Não avaliar conhecimento especializado, matemática ou programação. Um modelo pode ser cortês sem ser fiel à instrução; registrar separadamente.

In [ ]:
rng=random.Random(SEED)
blind=[];key=[]
for _,r in native.iterrows():
 blind_id=hashlib.sha256((signature+'|'+r['version']+'|'+r['task_id']+'|'+r['strategy']).encode()).hexdigest()[:16]
 blind.append({'blind_id':blind_id,'task_id':r['task_id'],'lang':r['lang'],'category':r['category'],
  'strategy':r['strategy'],'prompt':r['prompt'],'response':r['response'],
  'relevance_0_2':'','clarity_0_2':'','logic_0_2':'','instruction_0_2':'',
  'conversation_0_2':'','language_0_2':'','uncertainty_0_2':'',
  'problematic_repetition_0_2':'','overall_0_2':'','notes':'','reviewer':''})
 key.append({'blind_id':blind_id,'version':r['version'],'task_id':r['task_id'],
             'strategy':r['strategy'],'format':r['format']})
rng.shuffle(blind)
pd.DataFrame(blind).to_csv(RUN_DIR/'human_review_blind.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(key).to_csv(RUN_DIR/'blind_key_KEEP_SEPARATE.csv',index=False,encoding='utf-8-sig')
print('Formulário cego:',len(blind),'| chave mantida separadamente')


## 7. Relatório e interpretação

In [ ]:
report={
 'title':'Maia Lite 0.3.0 — Chat, língua e lógica',
 'signature':signature,'status':'AUTOMATIC_COMPLETE_HUMAN_REVIEW_PENDING',
 'versions':VERSIONS,'task_count':len(selected),'native_rows':len(native),'total_rows':len(df),
 'native_summary':json.loads(summary.to_json(orient='records')),
 'format_stress_summary':json.loads(stress.to_json(orient='records')),
 'limitations':[
  'Sem avaliação de matemática, programação ou conhecimento especializado.',
  'Acertos exatos não medem compreensão profunda; paráfrases corretas podem ser penalizadas.',
  'Repetição e EOS não medem qualidade factual ou utilidade.',
  'Teste de contexto é transcrição textual, não memória persistente.',
  'O formato nativo difere entre v2.2 e v2.3; comparar ambos como sistemas completos.',
  'Não inferir melhora estatística sem múltiplas amostras e intervalos de confiança.',
  'Amostra rápida é somente teste funcional; não é benchmark definitivo.'
 ]}
(RUN_DIR/'report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
lines=['# Maia Lite — Avaliação de chat, língua e lógica','',
 f'- Assinatura: `{signature}`',f'- Tarefas: {len(selected)}',
 f'- Respostas no formato nativo: {len(native)}',
 '- Revisão humana: **pendente**','',
 '## Como ler os resultados','',
 'Comparar `native_summary.csv` para capacidade conversacional no formato de cada adaptador.',
 'Usar `format_stress_summary.csv` somente para robustez ao delimitador.',
 'Verificar respostas em `generations_scored.csv` e preencher `human_review_blind.csv`.',
 'Não promover uma versão com base apenas em EOS, loss ou acerto exato.','',
 '## Limitações','']+['- '+x for x in report['limitations']]
(RUN_DIR/'report.md').write_text('\n'.join(lines),encoding='utf-8')
print('Relatórios salvos em:',RUN_DIR)
for p in sorted(RUN_DIR.iterdir()):print('-',p.name)


## 8. Próximos passos após a execução

1. Execute primeiro em **QUICK_MODE=True** para validar os caminhos, a arquitetura e a geração nos três idiomas.
2. Depois use `QUICK_MODE=False` e `FULL_MODE=True` e execute novamente a partir da configuração (a assinatura de execução criará outra pasta).
3. Envie `report.json`, `native_summary.csv`, `format_stress_summary.csv` e `generations_scored.csv` para análise; a revisão cega é recomendada antes de qualquer decisão de SFT.
4. Não treine v2.4 apenas para melhorar esses testes: eles são um instrumento de diagnóstico e devem ser complementados por exemplos não vistos.

**Observação:** o teste rápido cobre 18 perguntas, enquanto o completo cobre 42. Com `RUN_FORMAT_STRESS=True`, são geradas respostas adicionais para analisar o separador; essas respostas não entram na pontuação principal.